# Chapter 7: Flowlines, Risers, and Pipeline Hydraulics

This notebook demonstrates pipeline hydraulic calculations using NeqSim's
**PipeBeggsAndBrills** model for multiphase flow in oil and gas production systems.

**Topics covered:**
- Multiphase gas-oil pipeline pressure drop modeling
- Effect of pipeline diameter on pressure drop
- Flow rate vs pressure drop relationship
- Liquid holdup behavior with gas velocity
- Pipeline capacity curves for sizing

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


In [2]:
import matplotlib.pyplot as plt
import numpy as np

from neqsim import jneqsim

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
Stream = jneqsim.process.equipment.stream.Stream
PipeBeggsAndBrills = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

## 7.1 Create Multiphase Gas-Oil Fluid

We model a typical wet gas / gas-condensate production fluid with light
hydrocarbons and a small amount of condensate (nC10) at 60 °C and 80 bara.

In [3]:
def create_pipeline_fluid(T_C=60.0, P_bara=80.0):
    """Create a gas-condensate fluid for pipeline calculations."""
    fluid = SystemSrkEos(273.15 + T_C, P_bara)
    fluid.addComponent("nitrogen", 0.02)
    fluid.addComponent("CO2", 0.03)
    fluid.addComponent("methane", 0.75)
    fluid.addComponent("ethane", 0.08)
    fluid.addComponent("propane", 0.05)
    fluid.addComponent("i-butane", 0.02)
    fluid.addComponent("n-butane", 0.02)
    fluid.addComponent("n-pentane", 0.01)
    fluid.addComponent("nC10", 0.02)
    fluid.setMixingRule("classic")
    fluid.setMultiPhaseCheck(True)
    return fluid

# Quick test
test_fluid = create_pipeline_fluid()
ops = ThermodynamicOperations(test_fluid)
ops.TPflash()
test_fluid.initProperties()
print(f"Number of phases: {test_fluid.getNumberOfPhases()}")
print(f"Gas density: {test_fluid.getPhase('gas').getDensity('kg/m3'):.2f} kg/m3")

Number of phases: 2
Gas density: 72.90 kg/m3


## 7.2 Figure 1: Pressure Drop vs Pipeline Diameter

At a fixed flow rate, larger diameter pipelines produce lower pressure drops.
This is the fundamental trade-off in pipeline sizing: larger pipe costs more
but reduces compression requirements.

In [4]:
diameters_inch = np.array([4, 6, 8, 10, 12, 14, 16])
diameters_m = diameters_inch * 0.0254  # Convert to meters
pressure_drops = []
flow_rate_kghr = 50000.0  # Fixed flow rate
pipe_length = 10000.0  # 10 km pipeline

for diam_m in diameters_m:
    fluid = create_pipeline_fluid(60.0, 80.0)
    feed = Stream("feed", fluid)
    feed.setFlowRate(flow_rate_kghr, "kg/hr")
    feed.setTemperature(273.15 + 60.0, "K")
    feed.setPressure(80.0, "bara")

    pipe = PipeBeggsAndBrills("pipeline", feed)
    pipe.setDiameter(diam_m)
    pipe.setLength(pipe_length)
    pipe.setElevation(0.0)
    pipe.setNumberOfIncrements(20)

    process = ProcessSystem()
    process.add(feed)
    process.add(pipe)
    try:
        process.run()
        dp = pipe.getPressureDrop()
        pressure_drops.append(float(dp))
    except Exception as e:
        print(f"  Diameter {diam_m*39.37:.0f} inch failed: {e}")
        pressure_drops.append(float('nan'))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(diameters_inch, pressure_drops, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Pipeline Inner Diameter (inches)', fontsize=12)
ax.set_ylabel('Pressure Drop (bar)', fontsize=12)
ax.set_title(f'Pressure Drop vs Pipeline Diameter\n'
             f'(Flow = {flow_rate_kghr/1000:.0f} t/hr, L = {pipe_length/1000:.0f} km, horizontal)',
             fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xlim(3, 17)
plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_42756\1794182842.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** Pressure drop decreases sharply with increasing diameter,
approximately following a $\Delta P \propto D^{-5}$ relationship for turbulent flow.
Going from 6-inch to 10-inch pipe typically reduces pressure drop by an order of
magnitude. This trade-off between pipe cost and operating pressure is central to
pipeline design.

## 7.3 Figure 2: Pressure Drop vs Flow Rate

For a fixed pipeline diameter, pressure drop increases non-linearly with flow rate,
roughly as $\Delta P \propto Q^{1.8}$ in the turbulent regime.

In [5]:
flow_rates_kghr = np.linspace(10000, 100000, 10)
dp_vs_flow = []
pipe_diam_m = 0.2032  # 8 inch

for flow in flow_rates_kghr:
    fluid = create_pipeline_fluid(60.0, 80.0)
    feed = Stream("feed", fluid)
    feed.setFlowRate(float(flow), "kg/hr")
    feed.setTemperature(273.15 + 60.0, "K")
    feed.setPressure(80.0, "bara")

    pipe = PipeBeggsAndBrills("pipeline", feed)
    pipe.setDiameter(pipe_diam_m)
    pipe.setLength(pipe_length)
    pipe.setElevation(0.0)
    pipe.setNumberOfIncrements(20)

    process = ProcessSystem()
    process.add(feed)
    process.add(pipe)
    try:
        process.run()
        dp_vs_flow.append(float(pipe.getPressureDrop()))
    except Exception:
        dp_vs_flow.append(float('nan'))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(flow_rates_kghr / 1000, dp_vs_flow, 'rs-', linewidth=2, markersize=7)
ax.set_xlabel('Flow Rate (tonnes/hr)', fontsize=12)
ax.set_ylabel('Pressure Drop (bar)', fontsize=12)
ax.set_title(f'Pressure Drop vs Flow Rate\n'
             f'(Diameter = 8 inch, L = {pipe_length/1000:.0f} km, horizontal)',
             fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_42756\3663654175.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** The non-linear increase in pressure drop with flow rate illustrates
why production systems often become pipeline-limited. Doubling the flow rate more
than doubles the pressure drop, which can lead to capacity constraints when
available driving pressure is limited.

## 7.4 Figure 3: Liquid Holdup vs Gas Superficial Velocity

As gas velocity increases, the liquid holdup decreases because the gas phase
carries liquid droplets more efficiently. We vary the gas-to-liquid ratio to
demonstrate this effect.

In [6]:
# Vary flow rate in a fixed-diameter pipe to change superficial velocities
flow_rates_sweep = np.linspace(10000, 120000, 12)
holdups = []
gas_velocities = []

for flow in flow_rates_sweep:
    fluid = create_pipeline_fluid(60.0, 80.0)
    feed = Stream("feed", fluid)
    feed.setFlowRate(float(flow), "kg/hr")
    feed.setTemperature(273.15 + 60.0, "K")
    feed.setPressure(80.0, "bara")

    pipe = PipeBeggsAndBrills("pipeline", feed)
    pipe.setDiameter(0.2032)  # 8 inch
    pipe.setLength(5000.0)  # 5 km
    pipe.setElevation(0.0)
    pipe.setNumberOfIncrements(10)

    process = ProcessSystem()
    process.add(feed)
    process.add(pipe)
    try:
        process.run()
        # Get holdup profile and take the average
        holdup_profile = list(pipe.getLiquidHoldupProfileList())
        if len(holdup_profile) > 0:
            avg_holdup = sum(float(h) for h in holdup_profile) / len(holdup_profile)
        else:
            avg_holdup = float('nan')
        holdups.append(avg_holdup)

        # Estimate superficial gas velocity from outlet stream
        outlet = pipe.getOutletStream()
        outlet_fluid = outlet.getThermoSystem()
        outlet_fluid.initProperties()
        if outlet_fluid.hasPhaseType("gas"):
            gas_density = float(outlet_fluid.getPhase("gas").getDensity("kg/m3"))
            total_flow_kgs = float(flow) / 3600.0
            # Approximate gas mass fraction from phase volumes
            gas_vol_frac = 1.0 - avg_holdup if avg_holdup < 1.0 else 0.01
            area = 3.14159 * (0.2032 / 2.0) ** 2
            sup_gas_vel = (total_flow_kgs / gas_density) / area * gas_vol_frac
            gas_velocities.append(sup_gas_vel)
        else:
            gas_velocities.append(float('nan'))
    except Exception:
        holdups.append(float('nan'))
        gas_velocities.append(float('nan'))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(gas_velocities, holdups, 'g^-', linewidth=2, markersize=8)
ax.set_xlabel('Approximate Superficial Gas Velocity (m/s)', fontsize=12)
ax.set_ylabel('Average Liquid Holdup (-)', fontsize=12)
ax.set_title('Liquid Holdup vs Gas Velocity\n(8-inch pipe, 5 km, horizontal)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_42756\235879571.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** As gas velocity increases, liquid holdup decreases because faster
gas entrains and carries liquid more efficiently. At low gas velocities, liquid
accumulates at the pipe bottom (stratified flow), while at high velocities,
the flow transitions to annular or mist flow with minimal holdup. This has
important implications for pipeline pigging, slugging, and terrain-induced
liquid accumulation.

## 7.5 Figure 4: Pipeline Capacity Curve

For a given available pressure differential (e.g., 20 bar), what is the
maximum throughput for each pipeline diameter? This is the fundamental
capacity curve used in pipeline sizing.

In [7]:
available_dp = 20.0  # Available pressure drop (bar)
diameters_cap_inch = np.array([6, 8, 10, 12, 14, 16])
diameters_cap_m = diameters_cap_inch * 0.0254
max_flows = []

for diam_m in diameters_cap_m:
    # Binary search for flow rate that gives available_dp
    low_flow, high_flow = 5000.0, 300000.0
    best_flow = low_flow
    for _ in range(15):  # 15 iterations for convergence
        mid_flow = (low_flow + high_flow) / 2.0
        fluid = create_pipeline_fluid(60.0, 80.0)
        feed = Stream("feed", fluid)
        feed.setFlowRate(mid_flow, "kg/hr")
        feed.setTemperature(273.15 + 60.0, "K")
        feed.setPressure(80.0, "bara")

        pipe = PipeBeggsAndBrills("pipeline", feed)
        pipe.setDiameter(float(diam_m))
        pipe.setLength(pipe_length)
        pipe.setElevation(0.0)
        pipe.setNumberOfIncrements(15)

        process = ProcessSystem()
        process.add(feed)
        process.add(pipe)
        try:
            process.run()
            dp = float(pipe.getPressureDrop())
            if dp < available_dp:
                low_flow = mid_flow
                best_flow = mid_flow
            else:
                high_flow = mid_flow
        except Exception:
            high_flow = mid_flow
    max_flows.append(best_flow / 1000.0)  # Convert to tonnes/hr

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(diameters_cap_inch.astype(str), max_flows, color='steelblue', edgecolor='navy', width=0.6)
ax.set_xlabel('Pipeline Diameter (inches)', fontsize=12)
ax.set_ylabel('Maximum Throughput (tonnes/hr)', fontsize=12)
ax.set_title(f'Pipeline Capacity Curve\n'
             f'(Available ΔP = {available_dp:.0f} bar, L = {pipe_length/1000:.0f} km)',
             fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, v in enumerate(max_flows):
    ax.text(i, v + max(max_flows)*0.02, f'{v:.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_42756\2327085072.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** The capacity curve shows that pipeline throughput increases
dramatically with diameter — approximately as $Q_{max} \propto D^{2.5}$.
Doubling the pipe diameter from 8 to 16 inches can increase capacity by roughly
5-6 times. This is why pipeline sizing is one of the most impactful early
decisions in field development, as it directly determines the peak production
rate the system can sustain.

## Summary

Key takeaways from this chapter:

1. **Pressure drop scales strongly with diameter** — small increases in pipe size
   yield large reductions in friction loss.
2. **Pressure drop is non-linear with flow rate** — systems become pressure-limited
   well before geometric limits are reached.
3. **Liquid holdup decreases with gas velocity** — higher gas rates improve liquid
   transport but may cause erosion and vibration.
4. **Pipeline capacity** is determined by the interplay of available pressure
   differential, fluid properties, diameter, and length.